# Silver Claim Implementation

Reads unmerged bronze claims, cleans and validates them, MERGEs valid rows into silver,
quarantines bad rows, and flags processed bronze rows.

| Outcome | Destination | Bronze `merge_flag` |
|---|---|---|
| Valid | `silver.claim` | `true` |
| Hard reject (bad data) | `silver.quarantine_claim` | `true` |
| Soft reject: policy not (yet) in `silver.policydata` | stays in bronze | `false` (retried next run) |

**Job order:** depends on Silver_Policy.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.claim (
  claim_id             INT,
  policy_id            INT,
  date_of_claim        DATE,
  claim_amount         DECIMAL(18,0),
  claim_status         STRING,
  LastUpdatedTimeStamp TIMESTAMP,
  merged_timestamp     TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/claim';

CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.quarantine_claim (
  claim_id              INT,
  policy_id             INT,
  date_of_claim         DATE,
  claim_amount          DECIMAL(18,0),
  claim_status          STRING,
  LastUpdatedTimeStamp  TIMESTAMP,
  reject_reason         STRING,
  quarantined_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/quarantine/claim';

In [0]:
# DECIMAL(18,0) drops anything after the decimal point. If this prints rows, the source has cents and
# the silver column should be DECIMAL(18,2) instead (change the table definition before the next load).
display(spark.sql("""
SELECT count(*) AS bronze_amounts_with_decimals
FROM policysystemdatabricks.bronze.claim
WHERE claim_amount IS NOT NULL AND CAST(claim_amount AS DOUBLE) != floor(CAST(claim_amount AS DOUBLE))
"""))

bronze_amounts_with_decimals
0


In [0]:
from pyspark.sql import functions as F, Window

COLUMNS = ["claim_id", "policy_id", "date_of_claim", "claim_amount", "claim_status", "LastUpdatedTimeStamp"]
SOFT_REASONS = ["missing_policy"]  # parent may arrive later -> retry instead of quarantine

# 1. Read only unmerged bronze claims
claim_df = spark.sql("SELECT * FROM policysystemdatabricks.bronze.claim WHERE merge_flag = false")

# 2. Clean: trim claim_status; date_of_claim as DATE (format MM-dd-yyyy only when displaying)
clean_df = claim_df.selectExpr(
    "claim_id",
    "policy_id",
    "to_date(date_of_claim) AS date_of_claim",
    "claim_amount",
    r"nullif(regexp_replace(trim(claim_status), '\\s+', ' '), '') AS claim_status",
    "LastUpdatedTimeStamp",
)

# 3. Deduplicate: one row per claim_id (latest LastUpdatedTimeStamp). Extra sort keys make the pick deterministic.
#    NULL claim_id rows are kept individually so each one is quarantined.
w = Window.partitionBy("claim_id").orderBy(
    F.col("LastUpdatedTimeStamp").desc_nulls_last(), F.col("claim_status"), F.col("claim_amount").desc_nulls_last())
df_result = (clean_df.withColumn("rn", F.row_number().over(w))
             .filter("rn = 1 OR claim_id IS NULL")
             .drop("rn"))

In [0]:
# Referential check against SILVER policies: a claim must not point to a policy that silver rejected.
silver_policies = (spark.table("policysystemdatabricks.silver.policydata")
                   .select("policy_id").distinct()
                   .withColumn("_policy_exists", F.lit(True)))

reject_reason = (
    F.when(F.col("claim_id").isNull(), "null_claim_id")
     .when(F.col("policy_id").isNull(), "null_policy_id")
     .when(F.col("date_of_claim").isNull(), "missing_date_of_claim")
     .when(F.col("claim_status").isNull(), "null_claim_status")
     .when(F.col("claim_amount").isNull(), "null_claim_amount")
     .when(F.col("claim_amount") <= 0, "non_positive_claim_amount")
     .when(F.col("LastUpdatedTimeStamp").isNull(), "null_last_updated_timestamp")
     .when(F.col("_policy_exists").isNull(), "missing_policy")
)

classified_df = (df_result
                 .join(silver_policies, on="policy_id", how="left")
                 .withColumn("reject_reason", reject_reason)
                 .select(*COLUMNS, "reject_reason"))
classified_df.createOrReplaceTempView("claim_classified")

display(classified_df.groupBy(F.coalesce("reject_reason", F.lit("VALID")).alias("outcome")).count())

outcome,count


In [0]:
display(spark.sql("""
MERGE INTO policysystemdatabricks.silver.claim AS t
USING (SELECT * FROM claim_classified WHERE reject_reason IS NULL) AS s
ON t.claim_id = s.claim_id
WHEN MATCHED THEN UPDATE SET
  t.policy_id            = s.policy_id,
  t.date_of_claim        = s.date_of_claim,
  t.claim_amount         = s.claim_amount,
  t.claim_status         = s.claim_status,
  t.LastUpdatedTimeStamp = s.LastUpdatedTimeStamp,
  t.merged_timestamp     = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (claim_id, policy_id, date_of_claim, claim_amount, claim_status, LastUpdatedTimeStamp, merged_timestamp)
  VALUES (s.claim_id, s.policy_id, s.date_of_claim, s.claim_amount, s.claim_status, s.LastUpdatedTimeStamp, current_timestamp())
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
soft_list = ", ".join(f"'{r}'" for r in SOFT_REASONS)

display(spark.sql(f"""
INSERT INTO policysystemdatabricks.silver.quarantine_claim
SELECT {", ".join(COLUMNS)}, reject_reason, current_timestamp()
FROM claim_classified
WHERE reject_reason IS NOT NULL AND reject_reason NOT IN ({soft_list})
"""))

num_affected_rows,num_inserted_rows
0,0


In [0]:
# The timestamp guard avoids flagging a newer bronze version that landed after this run read the table.
display(spark.sql(f"""
MERGE INTO policysystemdatabricks.bronze.claim AS b
USING (SELECT claim_id, LastUpdatedTimeStamp FROM claim_classified
       WHERE claim_id IS NOT NULL
         AND (reject_reason IS NULL OR reject_reason NOT IN ({soft_list}))) AS s
ON  b.claim_id = s.claim_id
AND b.merge_flag = false
AND (s.LastUpdatedTimeStamp IS NULL OR b.LastUpdatedTimeStamp IS NULL
     OR b.LastUpdatedTimeStamp <= s.LastUpdatedTimeStamp)
WHEN MATCHED THEN UPDATE SET b.merge_flag = true
"""))

# NULL keys cannot be matched in a MERGE; they are always quarantined above, so flag them directly.
display(spark.sql("""
UPDATE policysystemdatabricks.bronze.claim
SET merge_flag = true
WHERE merge_flag = false AND claim_id IS NULL
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


num_affected_rows
0


In [0]:
%sql
SELECT (SELECT count(*) FROM policysystemdatabricks.bronze.claim WHERE merge_flag = false) AS bronze_waiting_for_policy,
       (SELECT count(*) FROM policysystemdatabricks.silver.quarantine_claim) AS quarantined_rows_total,
       count(*)                                   AS silver_rows,
       count(DISTINCT claim_id)                   AS silver_distinct_claims,
       count_if(claim_id IS NULL OR policy_id IS NULL OR claim_status IS NULL OR date_of_claim IS NULL
                OR claim_amount IS NULL OR LastUpdatedTimeStamp IS NULL) AS rows_with_nulls,  -- should be 0
       count_if(claim_amount <= 0)                AS non_positive_amounts,                    -- should be 0
       (SELECT count(*) FROM policysystemdatabricks.silver.claim c
        LEFT ANTI JOIN policysystemdatabricks.silver.policydata p ON c.policy_id = p.policy_id) AS claims_without_silver_policy,  -- should be 0
       count_if(merged_timestamp IS NULL)         AS rows_missing_merged_timestamp,           -- should be 0
       min(date_format(date_of_claim, 'MM-dd-yyyy')) AS sample_date_of_claim
FROM policysystemdatabricks.silver.claim

bronze_waiting_for_policy,quarantined_rows_total,silver_rows,silver_distinct_claims,rows_with_nulls,non_positive_amounts,claims_without_silver_policy,rows_missing_merged_timestamp,sample_date_of_claim
0,0,943,943,0,0,218,0,01-02-2023
